In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
df = pd.read_csv(
    "/kaggle/input/datasets/imdevskp/corona-virus-report/country_wise_latest.csv",
    usecols=["Confirmed", "Recovered", "New cases", "New recovered",
             "Confirmed last week", "1 week change", "1 week % increase",
             "WHO Region", "Deaths"]
)

In [ ]:
df.sample(10)

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
X = df.drop(columns=["Deaths"])
X = pd.get_dummies(X, columns=["WHO Region"], drop_first=True)
y = df["Deaths"]

In [ ]:
X.head()

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state=42)

In [ ]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

#Train Model

model = LinearRegression()
model.fit(X_train, y_train)

#Predict on test
y_pred = model.predict(X_test)

print(r2_score(y_test, y_pred))

In [ ]:
df["Deaths"].describe()

In [ ]:
df["Deaths"].sort_values(ascending=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.scatter(y_test,y_pred)
plt.xlabel("Actual Death")
plt.ylabel("Predicted Death")
plt.plot([y_test.min(), y_test.max()],[y_test.min(), y_test.max()], 'r--')
plt.show()

In [ ]:
y_log = np.log1p(df["Deaths"])

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train_log)

y_pred_log = model.predict(X_test)

# Convert back to real death counts before evaluating
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test_log)

print(r2_score(y_test_actual, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print(r2_score(y_test, rf_pred))

In [ ]:
from sklearn.metrics import mean_absolute_error
print(mean_absolute_error(np.expm1(y_test_log), np.expm1(ridge.predict(X_test))))


In [ ]:
from sklearn.linear_model import Ridge

# Diagnose: check how extreme the LinearRegression coefficients got
model.fit(X_train, y_train_log)
print("Linear coefficients:", model.coef_)

# Fixed model: Ridge handles multicollinearity much better
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train_log)

y_pred_log_ridge = ridge.predict(X_test)
y_pred_ridge = np.expm1(y_pred_log_ridge)
y_test_actual = np.expm1(y_test_log)

print("Ridge R² (real scale):", r2_score(y_test_actual, y_pred_ridge))
print("Ridge R² (log scale):", r2_score(y_test_log, y_pred_log_ridge))

In [ ]:
!git status